# 00 - Competition Intro And Data Setup

This notebook is the first pass through the Kaggle dataset.

Goals:
- load the raw competition files
- check shapes, columns, and date coverage
- inspect missing values
- inspect the target distribution and its outliers

Use this notebook before trying to model anything.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)

ROOT = Path.cwd().resolve().parent
DATA_DIR = ROOT / 'data' / 'raw'
TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'
SAMPLE_PATH = DATA_DIR / 'sample_submission.csv'

ROOT, DATA_DIR

(WindowsPath('C:/Users/joni0/kaggle-competition-stock-return-fundamentals'),
 WindowsPath('C:/Users/joni0/kaggle-competition-stock-return-fundamentals/data/raw'))

In [2]:
required_files = [TRAIN_PATH, TEST_PATH, SAMPLE_PATH]
missing_files = [path.name for path in required_files if not path.exists()]

if missing_files:
    raise FileNotFoundError(
        'Missing competition files in data/raw: ' + ', '.join(missing_files)
    )

print('All required files are present.')

All required files are present.


In [3]:
date_columns = ['period_start', 'period_end']

train = pd.read_csv(TRAIN_PATH, parse_dates=date_columns)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_PATH)

print('train shape:', train.shape)
print('test shape:', test.shape)
print('sample_submission shape:', sample_submission.shape)

print('\nTrain date range:')
print(train['period_start'].min(), 'to', train['period_start'].max())
print('Train forward return end range:')
print(train['period_end'].min(), 'to', train['period_end'].max())

print('\nTest start_year range:')
print(test['start_year'].min(), 'to', test['start_year'].max())

train shape: (23070, 39)
test shape: (8520, 36)
sample_submission shape: (8520, 2)

Train date range:
2019-01-01 00:00:00 to 2022-12-31 00:00:00
Train forward return end range:
2019-11-30 00:00:00 to 2023-12-31 00:00:00

Test start_year range:
2024 to 2024


In [4]:
display(train.head())
display(test.head())

print('Number of train columns:', len(train.columns))
print('Number of test columns:', len(test.columns))

print('\nColumn dtypes:')
display(train.dtypes.sort_index())

,id,ticker,start_year,period_start,period_end,return_pct,pe_ttm,price_to_book,price_to_sales,growth_pe_ratio,gross_margin,operating_margin,net_margin,roa,roe,rote,revenue_growth_3y,revenue_growth_yoy,revenue_ttm,net_income_ttm,income_before_tax,eps_basic,eps_diluted,total_assets,stockholders_equity,current_assets,current_liabilities,long_term_debt,goodwill,inventory,current_ratio,quick_ratio,debt_to_equity,dividend_yield,dividends_ttm,dividends_paid_ttm,shares_outstanding,shares_diluted,sector_code
0,0,AAPL,2022,2022-03-26,2023-04-01,-5.62,25.28,38.23,6.68,1.95,43.75,30.82,26.41,29.07,151.24,151.24,49.34,18.63,3.860170e+11,1.019350e+11,3.013900e+10,1.54,1.52,3.506620e+11,6.739900e+10,1.181800e+11,1.275080e+11,2.066460e+11,0.0,5.460000e+09,0.93,0.88,3.07,0.57,1.473400e+10,1.468700e+10,1.620757e+10,1.640332e+10,0.0
1,1,AAPL,2022,2022-06-25,2023-07-01,36.93,20.97,35.95,5.39,2.37,43.26,27.82,25.71,29.63,171.46,171.46,49.61,11.63,3.875420e+11,9.963300e+10,2.306600e+10,1.20,1.20,3.363090e+11,5.810700e+10,1.122920e+11,1.298730e+11,1.894000e+11,0.0,5.433000e+09,0.86,0.82,3.26,0.71,1.477800e+10,1.473400e+10,1.609538e+10,1.626220e+10,0.0
2,2,AAPL,2022,2022-09-24,2023-09-30,13.81,22.23,43.78,5.63,2.32,43.31,30.29,25.31,28.29,196.96,196.96,51.56,7.79,3.943280e+11,9.980300e+10,1.191030e+11,6.15,6.11,3.527550e+11,5.067200e+10,1.354050e+11,1.539820e+11,1.979180e+11,0.0,4.946000e+09,0.88,0.85,3.91,0.67,1.484100e+10,1.479300e+10,1.594342e+10,1.632582e+10,0.0
3,3,AAPL,2022,2022-12-31,2023-12-30,48.18,20.13,33.78,4.94,2.22,42.96,30.74,24.56,27.45,167.77,167.77,44.77,2.44,3.875370e+11,9.517100e+10,3.562300e+10,1.89,1.88,3.467470e+11,5.672700e+10,1.287770e+11,1.372860e+11,1.992540e+11,0.0,6.820000e+09,0.94,0.89,3.51,0.78,1.487700e+10,1.484000e+10,1.584241e+10,1.595572e+10,0.0
4,4,AAPL,2021,2021-03-27,2022-03-26,44.15,23.43,25.84,5.49,1.35,42.51,30.70,23.45,22.63,110.31,110.31,31.52,21.43,3.254060e+11,7.631100e+10,2.801100e+10,1.41,1.40,3.371580e+11,6.917800e+10,1.214650e+11,1.063850e+11,2.172840e+11,0.0,5.219000e+09,1.14,1.09,3.14,0.80,1.422700e+10,1.421200e+10,1.668630e+10,1.692916e+10,0.0


,id,ticker,start_year,pe_ttm,price_to_book,price_to_sales,growth_pe_ratio,gross_margin,operating_margin,net_margin,roa,roe,rote,revenue_growth_3y,revenue_growth_yoy,revenue_ttm,net_income_ttm,income_before_tax,eps_basic,eps_diluted,total_assets,stockholders_equity,current_assets,current_liabilities,long_term_debt,goodwill,inventory,current_ratio,quick_ratio,debt_to_equity,dividend_yield,dividends_ttm,dividends_paid_ttm,shares_outstanding,shares_diluted,sector_code
0,0,stock_0820,2024,NaN,NaN,NaN,NaN,38.62,-122.73,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-5435000.0,-0.21,-0.21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.546700e+07,0.0
1,1,stock_0238,2024,14.76,NaN,2.09,NaN,NaN,23.45,14.15,NaN,NaN,NaN,NaN,NaN,1.386100e+08,1.961600e+07,27451000.0,4.81,4.80,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.087000e+06,4.0
2,2,stock_0704,2024,25.46,0.98,1.74,0.76,NaN,NaN,6.85,0.47,3.85,4.09,19.47,-19.78,7.554160e+08,5.176500e+07,18008000.0,0.13,0.13,1.103751e+10,1.342895e+09,NaN,NaN,0.000000e+00,7.571700e+07,NaN,NaN,NaN,0.00,0.0,0.0,NaN,1.124020e+08,1.139223e+08,1.0
3,3,stock_0689,2024,14.43,1.34,1.54,-2.40,NaN,17.03,10.65,2.30,9.26,12.41,-34.58,14.36,2.275000e+10,2.422000e+09,494000000.0,0.45,0.45,1.051130e+11,2.615200e+10,8.557000e+09,8.275000e+09,8.607800e+10,6.630000e+09,294000000.0,1.03,1.00,3.29,NaN,NaN,NaN,1.000000e+09,1.001000e+09,9.0
4,4,stock_0006,2024,-316.67,45.39,12.36,NaN,82.05,1.50,-3.90,-3.21,-14.33,107.24,NaN,NaN,4.166111e+09,-1.625520e+08,19775000.0,0.05,0.05,5.063265e+09,1.134171e+09,2.913929e+09,2.468213e+09,1.849448e+09,1.285745e+09,NaN,1.18,1.18,1.63,NaN,NaN,NaN,NaN,2.617780e+08,0.0


Number of train columns: 39
Number of test columns: 36

Column dtypes:


current_assets                float64
current_liabilities           float64
current_ratio                 float64
debt_to_equity                float64
dividend_yield                float64
dividends_paid_ttm            float64
dividends_ttm                 float64
eps_basic                     float64
eps_diluted                   float64
goodwill                      float64
gross_margin                  float64
growth_pe_ratio               float64
id                              int64
income_before_tax             float64
inventory                     float64
long_term_debt                float64
net_income_ttm                float64
net_margin                    float64
operating_margin              float64
pe_ttm                        float64
period_end             datetime64[us]
period_start           datetime64[us]
price_to_book                 float64
price_to_sales                float64
quick_ratio                   float64
return_pct                    float64
revenue_grow

In [5]:
missing_summary = (
    train.isna()
    .mean()
    .sort_values(ascending=False)
    .rename('missing_rate')
    .to_frame()
)
missing_summary['missing_pct'] = 100 * missing_summary['missing_rate']

display(missing_summary.head(20))

plt.figure(figsize=(10, 8))
top_missing = missing_summary.head(20).sort_values('missing_pct')
plt.barh(top_missing.index, top_missing['missing_pct'])
plt.title('Top 20 Train Features By Missingness')
plt.xlabel('Missing percentage')
plt.tight_layout()
plt.show()

,missing_rate,missing_pct
dividends_paid_ttm,0.932163,93.216298
dividend_yield,0.720286,72.028609
dividends_ttm,0.708106,70.810577
gross_margin,0.624621,62.462072
inventory,0.499090,49.908973
debt_to_equity,0.372432,37.243173
shares_outstanding,0.353749,35.374946
growth_pe_ratio,0.347464,34.746424
income_before_tax,0.346121,34.612050
long_term_debt,0.330993,33.099263


C:\Users\joni0\AppData\Local\Temp\ipykernel_29396\2101223689.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
target = train['return_pct']

print(target.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(target, bins=80, ax=axes[0])
axes[0].set_title('Target Distribution: return_pct')
axes[0].set_xlabel('1-year forward return (%)')

sns.boxplot(x=target, ax=axes[1])
axes[1].set_title('Target Boxplot')
axes[1].set_xlabel('1-year forward return (%)')

plt.tight_layout()
plt.show()

count    23070.000000
mean        18.778736
std        138.650112
min        -99.170000
1%         -80.195500
5%         -56.198500
25%        -19.787500
50%          3.500000
75%         33.827500
95%        118.077000
99%        299.166000
max      10571.110000
Name: return_pct, dtype: float64


C:\Users\joni0\AppData\Local\Temp\ipykernel_29396\3715983722.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
train = train.assign(obs_year=train['start_year'])

year_summary = train.groupby('obs_year')['return_pct'].agg(['count', 'mean', 'median', 'std'])
display(year_summary)

plt.figure(figsize=(8, 5))
sns.boxplot(data=train, x='obs_year', y='return_pct')
plt.ylim(train['return_pct'].quantile(0.01), train['return_pct'].quantile(0.99))
plt.title('Target Distribution By Observation Year (1st to 99th pct clipped view)')
plt.xlabel('Observation year')
plt.ylabel('return_pct')
plt.tight_layout()
plt.show()

,count,mean,median,std
obs_year,,,,
2019,5029,4.098202,-7.700,86.715608
2020,5339,73.873791,42.790,254.149372
2021,6068,-10.513654,-12.155,41.156769
2022,6634,12.360606,3.260,64.771949


C:\Users\joni0\AppData\Local\Temp\ipykernel_29396\2227094178.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## What To Notice

- Missing values are part of the competition, not a data-cleaning mistake.
- The target is likely to be heavy-tailed, so RMSE can move a lot from a few names.
- Date-aware validation is mandatory because the test set begins in 2024.

Next: open `01_eda_and_validation_design.ipynb` and design a defensible validation scheme.